# 05 — Fine-tuning de T5 con `Seq2SeqTrainer`

> **Objetivo.** Entrenar `t5-small` sobre el subset GEC tokenizado, en MPS, usando `Seq2SeqTrainer` de Hugging Face.

## ¿Por qué `Seq2SeqTrainer` y no `Trainer`?

La guía técnica original sugiere `Trainer` con `predict_with_generate=True`. **Eso no compila** — el flag y los argumentos de generación solo existen en la subclase `Seq2SeqTrainingArguments`. Si usamos `Trainer` plano la evaluación calcula métricas sobre logits crudos en lugar del texto generado, lo que arruina BLEU/GLEU.

Decisión documentada en `docs/decisiones_tecnicas.md` ADR-04.

## Loop de entrenamiento — qué hace internamente

Para cada batch:
```
forward:        encoder(input_ids) → hidden
                decoder(hidden, shifted_labels) → logits
loss:           cross_entropy(logits, labels)        # labels=-100 enmascarados
backward:       loss.backward()
optimizer:      AdamW.step() → actualiza pesos
scheduler:      linear warmup + decay
```

Al final de cada epoch (configurable):
- **Eval:** generación auto-regresiva sobre eval set, calcula métricas.
- **Checkpoint:** guarda pesos en `output_dir/checkpoint-XXX`.


In [ ]:
# === CONFIG ===
MODEL_NAME      = "google-t5/t5-small"
BATCH_SIZE      = 8        # M1 8GB → 4, M1 16GB → 8, M2 Pro+ → 16
GRAD_ACCUM      = 2        # batch efectivo = BATCH_SIZE * GRAD_ACCUM
LR              = 3e-4     # T5 prefiere LR mayor que BERT (paper original)
NUM_EPOCHS      = 3
WEIGHT_DECAY    = 0.01
SEED            = 42
USE_BF16        = True     # Apple Silicon soporta bf16; bajar a False si crash
# ==============

import os, torch
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

from pathlib import Path
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    set_seed,
)

set_seed(SEED)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TOKENIZED = ROOT / "artifacts" / "tokenized_dataset"
OUTPUT_DIR = ROOT / "artifacts" / "t5-gec-mac"

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("BF16 :", USE_BF16)

## 1. Cargar tokenizer, modelo y dataset tokenizado

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

tokenized = load_from_disk(str(TOKENIZED))
print(tokenized)
print("Modelo:", MODEL_NAME, "-", f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

## 2. Métrica de evaluación durante el entrenamiento

Durante el fine-tuning queremos una señal *barata* — usamos **SacreBLEU**. Métricas más caras (GLEU, ERRANT) van en notebook 06.

La función `compute_metrics` recibe `(preds, labels)` ya en token-ids; debemos:
1. Reemplazar `-100` en labels con `pad_token_id` para poder decodificar.
2. Decodificar a strings.
3. Calcular SacreBLEU.

In [ ]:
import numpy as np
import evaluate

bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [[l.strip()] for l in decoded_labels]   # sacrebleu espera lista de refs

    score = bleu.compute(predictions=decoded_preds, references=decoded_labels)
    return {"sacrebleu": round(score["score"], 4)}

## 3. Argumentos de entrenamiento

Notas clave:
- `eval_strategy="epoch"` — el nombre nuevo de `evaluation_strategy` (renombrado en transformers ≥4.41).
- `predict_with_generate=True` — ejecuta `model.generate()` en eval para producir texto, no logits.
- `bf16=True` — Apple Silicon soporta bfloat16 nativo; preferir sobre fp16 en MPS.
- `use_mps_device` ya no se setea — auto-detectado.
- `report_to="none"` — evita warnings de wandb/tensorboard si no están instalados.

In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    overwrite_output_dir=True,

    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.05,
    lr_scheduler_type="linear",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="sacrebleu",
    greater_is_better=True,

    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,

    bf16=USE_BF16,
    fp16=False,                # MPS prefiere bf16
    dataloader_num_workers=2,

    logging_steps=50,
    report_to="none",
    seed=SEED,
)

## 4. Collator y Trainer

In [ ]:
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

## 5. Entrenar

En M1 Pro 16 GB con 9 000 ejemplos × 3 epochs ≈ **20-30 min**. Monitorea la GPU en *Monitor de Actividad → Ventana → GPU*.

In [ ]:
train_result = trainer.train()
print(train_result.metrics)

## 6. Guardar el mejor modelo + tokenizer

El mejor checkpoint (por SacreBLEU eval) ya está cargado por `load_best_model_at_end=True`. Lo persistimos como un único directorio `final/` para que los notebooks 06 y 07 lo carguen sin ambigüedad.

In [ ]:
FINAL = OUTPUT_DIR / "final"
trainer.save_model(str(FINAL))
tokenizer.save_pretrained(str(FINAL))
print("Modelo final guardado en:", FINAL)

## 7. Inferencia rápida — sanity check

Una oración con error obvio para verificar que aprendió algo.

In [ ]:
model.eval()
tests = [
    "He go to school every day.",
    "I has a apple.",
    "She don't like coffee.",
    "This is a correct sentence.",   # debería dejarse igual o casi
]

for t in tests:
    inputs = tokenizer("grammar: " + t, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=64, num_beams=4)
    print(f"  IN : {t}")
    print(f"  OUT: {tokenizer.decode(out[0], skip_special_tokens=True)}\n")

## Resumen

- Modelo entrenado y persistido en `artifacts/t5-gec-mac/final/`.
- Mejor SacreBLEU registrado en `train_result.metrics`.
- `load_best_model_at_end` garantiza que la versión guardada es la del mejor checkpoint.

**Siguiente → `06_evaluacion.ipynb`** para evaluación cuantitativa formal con SacreBLEU + GLEU + Exact-Match.
